# Debug GigaChat bind_tools

Проверяем, работает ли function_calling с вложенными Pydantic-моделями.

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser('~/ace_experiment/ace-gigachat'))

from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage
import json

In [ ]:
# --- Init GigaChat ---
gc = GigaChat(
    base_url='https://gigachat-ift.sberdevices.delta.sbrf.ru/v1',
    model='GigaChat-2-Max',
    verify_ssl_certs=False,
    cert_file='.certs/cert_agents.txt',
    key_file='.certs/key_agents.txt',
    profanity_check=False,
    repetition_penalty=1,
    timeout=120,
    streaming=False,
    temperature=0.0,
)
print('GigaChat initialized')

## Тест 1: Простая схема (flat, без вложенности)

In [ ]:
class SimpleOutput(BaseModel):
    """Simple flat output."""
    reasoning: str = Field(description='Цепочка рассуждений')
    answer: str = Field(description='Ответ')

bound = gc.bind_tools([SimpleOutput], tool_choice='SimpleOutput')
result = bound.invoke([HumanMessage(content='Сколько будет 2+2? Ответь кратко.')])

print('=== tool_calls ===')
print(result.tool_calls)
print()
print('=== content ===')
print(repr(result.content))
print()
print('=== additional_kwargs ===')
print(result.additional_kwargs)

## Тест 2: Вложенная схема (List[NestedModel])

In [ ]:
class Step(BaseModel):
    """Single step."""
    action: str = Field(description='Действие')
    detail: Optional[str] = Field(default=None, description='Детали')

class PlanOutput(BaseModel):
    """Plan with nested steps."""
    reasoning: str = Field(description='Рассуждения')
    steps: List[Step] = Field(description='Шаги плана')

bound = gc.bind_tools([PlanOutput], tool_choice='PlanOutput')
result = bound.invoke([HumanMessage(content='Составь план из 3 шагов как приготовить чай.')])

print('=== tool_calls ===')
print(json.dumps(result.tool_calls, indent=2, ensure_ascii=False) if result.tool_calls else 'EMPTY')
print()
print('=== content ===')
print(repr(result.content))
print()
print('=== additional_kwargs ===')
print(result.additional_kwargs)

## Тест 3: CuratorOutput (наша реальная схема)

In [ ]:
from gigachat_client import CuratorOutput, _CuratorOperation

bound = gc.bind_tools([CuratorOutput], tool_choice='CuratorOutput')

prompt = """Ты куратор плейбука. Добавь один пункт в секцию Инструкции о том, 
что при запросе справки на английском языке нужно уточнить формат.
Верни одну операцию ADD."""

result = bound.invoke([HumanMessage(content=prompt)])

print('=== tool_calls ===')
print(json.dumps(result.tool_calls, indent=2, ensure_ascii=False) if result.tool_calls else 'EMPTY')
print()
print('=== content ===')
print(repr(result.content))
print()
print('=== additional_kwargs ===')
print(result.additional_kwargs)

## Тест 4: CuratorOutput с List[dict] (старая схема без типизации)

In [ ]:
class CuratorOutputFlat(BaseModel):
    """Curator response — flat dict operations."""
    reasoning: str = Field(description='Анализ состояния плейбука')
    operations: List[dict] = Field(description='Список операций ADD/UPDATE/MERGE/ARCHIVE')

bound = gc.bind_tools([CuratorOutputFlat], tool_choice='CuratorOutputFlat')
result = bound.invoke([HumanMessage(content=prompt)])

print('=== tool_calls ===')
print(json.dumps(result.tool_calls, indent=2, ensure_ascii=False) if result.tool_calls else 'EMPTY')
print()
print('=== content ===')
print(repr(result.content))

## Тест 5: tool_choice варианты

Может tool_choice нужен в другом формате?

In [ ]:
# Вариант A: tool_choice как dict
try:
    bound = gc.bind_tools([CuratorOutput], tool_choice={'type': 'function', 'function': {'name': 'CuratorOutput'}})
    result = bound.invoke([HumanMessage(content=prompt)])
    print('Dict tool_choice: tool_calls =', bool(result.tool_calls))
    if result.tool_calls:
        print(json.dumps(result.tool_calls[0], indent=2, ensure_ascii=False))
except Exception as e:
    print(f'Dict tool_choice failed: {e}')

print()

# Вариант B: tool_choice = 'auto'
try:
    bound = gc.bind_tools([CuratorOutput], tool_choice='auto')
    result = bound.invoke([HumanMessage(content=prompt)])
    print('Auto tool_choice: tool_calls =', bool(result.tool_calls))
    if result.tool_calls:
        print(json.dumps(result.tool_calls[0], indent=2, ensure_ascii=False))
    else:
        print('content:', result.content[:200])
except Exception as e:
    print(f'Auto tool_choice failed: {e}')

print()

# Вариант C: tool_choice = 'any'
try:
    bound = gc.bind_tools([CuratorOutput], tool_choice='any')
    result = bound.invoke([HumanMessage(content=prompt)])
    print('Any tool_choice: tool_calls =', bool(result.tool_calls))
    if result.tool_calls:
        print(json.dumps(result.tool_calls[0], indent=2, ensure_ascii=False))
    else:
        print('content:', result.content[:200])
except Exception as e:
    print(f'Any tool_choice failed: {e}')

## Тест 6: Без bind_tools — with_structured_output

langchain-gigachat может поддерживать `with_structured_output` напрямую.

In [ ]:
try:
    structured = gc.with_structured_output(CuratorOutput)
    result = structured.invoke(prompt)
    print(f'Type: {type(result)}')
    print(f'Result: {result}')
    if isinstance(result, CuratorOutput):
        print(f'\noperations count: {len(result.operations)}')
        for op in result.operations:
            print(f'  type={op.type} section={op.section} content={op.content[:50] if op.content else None}')
except Exception as e:
    print(f'with_structured_output failed: {type(e).__name__}: {e}')

## Тест 7: Проверка JSON schema который отправляется в API

Посмотрим какую schema langchain генерирует из Pydantic-модели.

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

tool_schema = convert_to_openai_tool(CuratorOutput)
print(json.dumps(tool_schema, indent=2, ensure_ascii=False))

## Сводка

In [ ]:
print('Проверь:')
print('1. В каких тестах tool_calls НЕ пустой? (bind_tools работает)')
print('2. В каких tool_calls пустой но content содержит JSON? (fallback на текст)')
print('3. Тест 6 (with_structured_output) — если работает, это лучше bind_tools')
print('4. Тест 7 — JSON schema может быть слишком сложной для GigaChat')